In [ ]:
import os
import json
import glob
import math
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

# ----------------------------
# Config
# ----------------------------
RESULT_DIR = "./result"
OUT_DIR = "./analysis_out/ch5_2"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")

EVENT_ORDER = ["Continue", "SizeChange", "Merge", "Split"]
W_ORDER = [1, 2, 5, 10, 20, 50]

K_COMM = 3
HALIAS_NORM_DEN = math.log2(K_COMM)

N_BINS = 10

# ----------------------------
# Utilities
# ----------------------------
def ensure_dirs():
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(TAB_DIR, exist_ok=True)

def save_csv(df: pd.DataFrame, name: str):
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=False)
    print(f"[saved] {path}")

def save_txt(text: str, name: str):
    path = os.path.join(TAB_DIR, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[saved] {path}")

# Normal CDF for p-values (scipy optional)
try:
    from scipy.stats import norm
    def norm_cdf(x: float) -> float:
        return float(norm.cdf(x))
except Exception:
    def norm_cdf(x: float) -> float:
        return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# ----------------------------
# Load
# ----------------------------
def load_all_results(result_dir: str) -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    if not paths:
        raise FileNotFoundError(f"No JSON files found in {result_dir}")

    rows = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"Result file must contain a list of trials: {p}")

        for t in data:
            gt = t.get("ground_truth")
            ans = t.get("answer")

            # Backward compatibility
            gt = "Continue" if gt == "Stable" else gt
            ans = "Continue" if ans == "Stable" else ans

            rows.append({
                "source_file": os.path.basename(p),
                "subject_id": t.get("subject_id"),
                "trial_index": t.get("trial_index"),
                "W": t.get("W"),
                "ground_truth": gt,
                "answer": ans,
                "correct": t.get("correct"),
                "h_alias": t.get("h_alias"),
            })

    df = pd.DataFrame(rows)

    for c in ["trial_index", "W", "correct", "h_alias"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["subject_id"] = df["subject_id"].astype(str)
    df["ground_truth"] = df["ground_truth"].astype(str)

    df = df.dropna(subset=["subject_id", "correct", "h_alias", "W", "ground_truth"]).copy()
    df["correct"] = df["correct"].astype(int)

    df["halias_norm"] = df["h_alias"] / HALIAS_NORM_DEN

    # categorical order (nice for tables)
    df["ground_truth"] = pd.Categorical(df["ground_truth"], categories=EVENT_ORDER, ordered=True)
    df["W"] = pd.Categorical(df["W"], categories=W_ORDER, ordered=True)

    return df

# ----------------------------
# Binned descriptive curves
# ----------------------------
def bin_summary(df: pd.DataFrame, x: str, y: str, n_bins: int, group: Optional[str] = None) -> pd.DataFrame:
    tmp = df[[x, y] + ([group] if group else [])].dropna().copy()

    def summarize_one(sub: pd.DataFrame) -> pd.DataFrame:
        try:
            sub["bin"] = pd.qcut(sub[x], q=n_bins, duplicates="drop")
        except Exception:
            sub["bin"] = pd.cut(sub[x], bins=n_bins)

        g = sub.groupby("bin", dropna=False)
        out = g.agg(
            n=(y, "count"),
            x_mean=(x, "mean"),
            acc=(y, "mean"),
        ).reset_index(drop=True)

        p = out["acc"].astype(float)
        n = out["n"].astype(float).replace(0, np.nan)
        se = np.sqrt(p * (1 - p) / n)
        out["ci95_lo"] = (p - 1.96 * se).clip(0, 1)
        out["ci95_hi"] = (p + 1.96 * se).clip(0, 1)
        return out.sort_values("x_mean")

    if group:
        frames = []
        for gname, sub in tmp.groupby(group):
            out = summarize_one(sub)
            out[group] = gname
            frames.append(out)
        return pd.concat(frames, ignore_index=True)
    else:
        return summarize_one(tmp)

def plot_binned_curve(bin_df: pd.DataFrame, title: str, out_png: str):
    plt.figure()
    x = bin_df["x_mean"].values
    y = bin_df["acc"].values
    lo = bin_df["ci95_lo"].values
    hi = bin_df["ci95_hi"].values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, lo, hi, alpha=0.2)
    plt.title(title)
    plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.02)
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")

def plot_binned_curve_by_event(bin_df: pd.DataFrame, title: str, out_png: str):
    plt.figure()
    for ev in ["SizeChange", "Merge", "Split", "Continue"]:
        sub = bin_df[bin_df["ground_truth"] == ev]
        if sub.empty:
            continue
        plt.plot(sub["x_mean"].values, sub["acc"].values, marker="o", label=ev)
    plt.title(title)
    plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.02)
    plt.legend()
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")

# ----------------------------
# GEE models + summary
# ----------------------------
def fit_gee(formula: str, df: pd.DataFrame):
    return smf.gee(
        formula=formula,
        groups="subject_id",
        data=df,
        family=sm.families.Binomial()
    ).fit()

def summarize_gee(model, name: str) -> pd.DataFrame:
    params = model.params
    se = model.bse
    z = params / se
    p = np.array([2.0 * (1.0 - norm_cdf(abs(float(zi)))) for zi in z])

    out = pd.DataFrame({
        "term": params.index,
        "beta": params.values,
        "se": se.values,
        "z": z.values,
        "p": p,
        "ci95_lo": params.values - 1.96 * se.values,
        "ci95_hi": params.values + 1.96 * se.values,
    })
    out["OR"] = np.exp(out["beta"])
    out["OR_ci95_lo"] = np.exp(out["ci95_lo"])
    out["OR_ci95_hi"] = np.exp(out["ci95_hi"])
    out["model"] = name
    return out

def write_key_effect(summary_df: pd.DataFrame, term: str, outname: str):
    row = summary_df[summary_df["term"] == term]
    if row.empty:
        save_txt(f"Term not found: {term}\n", outname)
        return
    r = row.iloc[0]
    txt = (
        f"Key effect ({term}):\n"
        f"beta = {r['beta']:.6f}\n"
        f"SE   = {r['se']:.6f}\n"
        f"z    = {r['z']:.3f}\n"
        f"p    = {r['p']:.4g}\n"
        f"OR   = {r['OR']:.3f} (95% CI [{r['OR_ci95_lo']:.3f}, {r['OR_ci95_hi']:.3f}])\n"
    )
    save_txt(txt, outname)

# ----------------------------
# Main
# ----------------------------
def main():
    ensure_dirs()
    df = load_all_results(RESULT_DIR)

    overview = (
        f"N_subjects = {df['subject_id'].nunique()}\n"
        f"N_trials   = {len(df)}\n"
        f"Mean accuracy = {df['correct'].mean():.3f}\n"
        f"halias_norm range = [{df['halias_norm'].min():.3f}, {df['halias_norm'].max():.3f}]\n"
    )
    save_txt(overview, "table_5_2_overview.txt")

    # ---- Descriptive curves (all)
    bins_all = bin_summary(df, x="halias_norm", y="correct", n_bins=N_BINS)
    save_csv(bins_all, "table_5_2_binned_acc.csv")
    plot_binned_curve(
        bins_all,
        title=r"Accuracy vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_2_acc_vs_halias_binned.png"
    )

    bins_ev = bin_summary(df, x="halias_norm", y="correct", n_bins=max(6, N_BINS // 2), group="ground_truth")
    save_csv(bins_ev, "table_5_2_binned_acc_by_event.csv")
    plot_binned_curve_by_event(
        bins_ev,
        title=r"Accuracy vs $\tilde{H}_{alias}$ by event type (binned means)",
        out_png="fig_5_2_acc_vs_halias_binned_by_event.png"
    )

    # ---- GEE main + controls (all events)
    m1 = fit_gee("correct ~ halias_norm", df)
    m2 = fit_gee("correct ~ halias_norm + C(ground_truth) + C(W)", df)

    s1 = summarize_gee(m1, "GEE_main: correct ~ halias_norm")
    s2 = summarize_gee(m2, "GEE_controls: + event + W")
    save_csv(s1, "table_5_2_gee_main.csv")
    save_csv(s2, "table_5_2_gee_controls.csv")
    write_key_effect(s1, "halias_norm", "table_5_2_key_effect_main.txt")
    write_key_effect(s2, "halias_norm", "table_5_2_key_effect_controls.txt")

    # ---- Sensitivity: exclude Continue
    df_nc = df[df["ground_truth"] != "Continue"].copy()
    if len(df_nc) > 0:
        m1_nc = fit_gee("correct ~ halias_norm", df_nc)
        m2_nc = fit_gee("correct ~ halias_norm + C(ground_truth) + C(W)", df_nc)

        s1_nc = summarize_gee(m1_nc, "GEE_main_noContinue")
        s2_nc = summarize_gee(m2_nc, "GEE_controls_noContinue")
        save_csv(s1_nc, "table_5_2_gee_main_noContinue.csv")
        save_csv(s2_nc, "table_5_2_gee_controls_noContinue.csv")
        write_key_effect(s1_nc, "halias_norm", "table_5_2_key_effect_main_noContinue.txt")
        write_key_effect(s2_nc, "halias_norm", "table_5_2_key_effect_controls_noContinue.txt")

    print("\n✅ 5.2 analysis complete.")
    print(f"Outputs in: {OUT_DIR}")

if __name__ == "__main__":
    main()